# 01.5 - 01.8 — Reshape, batched matmul, einsum, reductions

**The question this block answers:** how do I move data between shapes without silently
scrambling it — and how do I write a contraction so a mistake becomes an error?

**Prereqs:** 00.5 (transpose, strides), 01.2-01.4.

**Interview one-liners**
- `transpose`/`permute` reorder **strides**, not data. The tensor is "wrong" in memory afterwards, which is exactly why `view` refuses.
- `view` reinterprets an existing contiguous layout and **fails loudly**. `reshape` is "view if possible, copy otherwise" — convenient, and a hidden allocation in a hot loop.
- The multi-head split is `view` **then** `transpose`. Doing a single `view` straight to `(B,H,T,d_head)` gives the right shape and the **wrong data**, with no error.
- On the way back you *must* `.contiguous()` before `view`, because you just transposed.
- In `einsum`, a letter that appears in the inputs but **not** in the output is the one summed over. The shape contract is written in the call, so a typo is an error rather than a broadcast.
- In a reduction, the axis you name in `dim=` is the axis that **disappears** — unless `keepdim=True`.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from common import assert_shape

torch.manual_seed(0)
print('torch', torch.__version__)

torch 2.10.0


---
# 01.5 — reshape vs view vs transpose, and contiguity

## Experiment 1 — strides, and what transpose really changed

> **Predict first:** after `transpose(1, 2)` on a `(2, 3, 4)` tensor, did any float move in
> memory? If not, what changed? Predict the new stride tuple before you print it.
>
> _(write your prediction here)_

In [2]:
x = torch.randn(2, 3, 4)
t = x.transpose(1, 2)

print('x  shape', tuple(x.shape), ' stride', x.stride(), ' contiguous', x.is_contiguous())
print('t  shape', tuple(t.shape), ' stride', t.stride(), ' contiguous', t.is_contiguous())
print('same storage?', x.data_ptr() == t.data_ptr())

x  shape (2, 3, 4)  stride (12, 4, 1)  contiguous True
t  shape (2, 4, 3)  stride (12, 1, 4)  contiguous False
same storage? True


### Reading the output
`(12, 4, 1)` became `(12, 1, 4)` — positions 1 and 2 swapped, nothing else. Same pointer.
**Not one float moved.**

A stride tells PyTorch how to turn an index tuple into a flat offset:
`offset = i*stride[0] + j*stride[1] + k*stride[2]`. Transpose just relabels which index uses
which stride. The data in memory is still laid out for the *old* shape — which is what
"not contiguous" means, and why the next cell fails.

In [3]:
try:
    t.view(2, 12)
except RuntimeError as e:
    print('t.view(2, 12)              -> RuntimeError:', str(e).split('\n')[0][:95])

print('t.reshape(2, 12)           ->', tuple(t.reshape(2, 12).shape), ' (copied silently)')
print('t.contiguous().view(2, 12) ->', tuple(t.contiguous().view(2, 12).shape), ' (you paid, explicitly)')
print()
print('did reshape copy?  new pointer:', t.reshape(2, 12).data_ptr() != t.data_ptr())

t.view(2, 12)              -> RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans a
t.reshape(2, 12)           -> (2, 12)  (copied silently)
t.contiguous().view(2, 12) -> (2, 12)  (you paid, explicitly)

did reshape copy?  new pointer: True


### Reading the output
- `view` **refuses** — it only reinterprets an existing contiguous layout, never copies.
- `reshape` **succeeds by copying**. Convenient, but that is a hidden allocation, and in a hot
  serving loop it is a real cost you did not write down.
- `.contiguous().view(...)` does the same thing with the copy made visible.

Prefer `view` while learning **precisely because it fails**. A loud error beats a silent copy.

## Experiment 2 — the multi-head split, right and wrong

This is the sequence you will write in Phase 3, and the one `PROGRESS.md` warns about:
*"the standard experience is getting a tensor of the right shape with the heads scrambled —
which raises no error and produces plausible numbers."*

Start from `(B=2, T=4, d_model=8)` and split into `H=2` heads of `d_head=4`.

```
CORRECT:  (B, T, d_model) --view--> (B, T, H, d_head) --transpose(1,2)--> (B, H, T, d_head)
WRONG:    (B, T, d_model) ----------------view---------------------------> (B, H, T, d_head)
```

Both end at `(2, 2, 4, 4)`. Only one is right.

In [4]:
B, T, d_model, H = 2, 4, 8, 2
d_head = d_model // H

x = torch.arange(B * T * d_model).float().reshape(B, T, d_model)

right = x.view(B, T, H, d_head).transpose(1, 2)   # (B,T,H,dh) -> (B,H,T,dh)
wrong = x.view(B, H, T, d_head)                   # straight there

assert_shape(right, (B, H, T, d_head), 'right')
assert_shape(wrong, (B, H, T, d_head), 'wrong')
print('both shapes:', tuple(right.shape), tuple(wrong.shape), ' - and no error from either\n')

print('token 0 of batch 0, all 8 features:', x[0, 0].tolist())
print()
print('RIGHT: head 0, batch 0 ->')
print(right[0, 0])
print('\nWRONG: head 0, batch 0 ->')
print(wrong[0, 0])

both shapes: (2, 2, 4, 4) (2, 2, 4, 4)  - and no error from either

token 0 of batch 0, all 8 features: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]

RIGHT: head 0, batch 0 ->
tensor([[ 0.,  1.,  2.,  3.],
        [ 8.,  9., 10., 11.],
        [16., 17., 18., 19.],
        [24., 25., 26., 27.]])

WRONG: head 0, batch 0 ->
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])


### Reading the output
Look at what each "head 0" contains.

**RIGHT** — each row is the *first 4 features* of a token: `[0,1,2,3]` from token 0,
`[8,9,10,11]` from token 1, and so on. A head sees **a slice of the feature vector, for every
token**. That is what a head is.

**WRONG** — the first row is `[0,1,2,3]` but the second is `[4,5,6,7]`, which is the *second
half of token 0*. This "head" has eaten tokens 0 and 1 whole and calls them four timesteps.
The time axis and the feature axis have been mixed together.

Attention over that runs fine, trains, and produces a model that is quietly broken.

**The rule: `view` may only split or merge *adjacent* axes. Moving an axis is `transpose`'s job.**
Splitting `d_model` into `(H, d_head)` is legal because they are adjacent; jumping `H` in front
of `T` is not.

In [5]:
# verify the claim rather than trusting the eyeball
print('right[b,h,t,:] == x[b, t, h*dh:(h+1)*dh] for every b,h,t ?')
ok = all(
    torch.equal(right[b, h, t], x[b, t, h * d_head:(h + 1) * d_head])
    for b in range(B) for h in range(H) for t in range(T)
)
print('  ', ok)

print('\nsame check for the wrong version:')
ok_w = all(
    torch.equal(wrong[b, h, t], x[b, t, h * d_head:(h + 1) * d_head])
    for b in range(B) for h in range(H) for t in range(T)
)
print('  ', ok_w)

right[b,h,t,:] == x[b, t, h*dh:(h+1)*dh] for every b,h,t ?
   True

same check for the wrong version:
   False


## Experiment 3 — the merge back, and why `.contiguous()` is mandatory

After attention you have `(B, H, T, d_head)` and need `(B, T, d_model)` again.

In [6]:
back = right.transpose(1, 2)          # (B,H,T,dh) -> (B,T,H,dh)
print('after transpose  ', tuple(back.shape), ' contiguous', back.is_contiguous())

try:
    back.view(B, T, d_model)
except RuntimeError as e:
    print('back.view(...)   -> RuntimeError:', str(e).split('\n')[0][:80])

merged = back.contiguous().view(B, T, d_model)
print('contiguous().view ->', tuple(merged.shape))
print('\nround-trip identical to the original x?', torch.equal(merged, x))

after transpose   (2, 4, 2, 4)  contiguous True
contiguous().view -> (2, 4, 8)

round-trip identical to the original x? True


### Reading the output
`True` — the split and merge are exact inverses, **provided** you `.contiguous()` on the way back.

Why mandatory on the return and not on the way out: the outward `view` acts on `x`, which was
contiguous. The return `view` acts on a tensor you *just transposed*, which is not. Same
operation, different starting condition.

---
# 01.6 — matmul and batched matmul semantics

`torch.matmul` behaves differently for 1-D, 2-D and >2-D inputs. The 1-D rules are where
people get surprised.

In [7]:
cases = [
    ('1-D @ 1-D', (3,),        (3,)),        # inner product -> scalar
    ('1-D @ 2-D', (3,),        (3, 5)),      # promoted to (1,3), then the 1 is removed
    ('2-D @ 1-D', (4, 3),      (3,)),        # promoted to (3,1), then the 1 is removed
    ('2-D @ 2-D', (4, 3),      (3, 5)),      # ordinary matmul
    ('3-D @ 2-D', (2, 4, 3),   (3, 5)),      # batch of 2
    ('4-D @ 4-D', (2, 2, 4, 3), (2, 2, 3, 5)),
    ('broadcast ', (2, 1, 4, 3), (5, 3, 6)),  # batch dims broadcast
]
for name, sa, sb in cases:
    out = torch.matmul(torch.randn(*sa), torch.randn(*sb))
    print(f'{name}  {str(sa):13s} @ {str(sb):12s} -> {tuple(out.shape) if out.ndim else "scalar"}')

1-D @ 1-D  (3,)          @ (3,)         -> scalar
1-D @ 2-D  (3,)          @ (3, 5)       -> (5,)
2-D @ 1-D  (4, 3)        @ (3,)         -> (4,)
2-D @ 2-D  (4, 3)        @ (3, 5)       -> (4, 5)
3-D @ 2-D  (2, 4, 3)     @ (3, 5)       -> (2, 4, 5)
4-D @ 4-D  (2, 2, 4, 3)  @ (2, 2, 3, 5) -> (2, 2, 4, 5)
broadcast   (2, 1, 4, 3)  @ (5, 3, 6)    -> (2, 5, 4, 6)


### Reading the output
The 1-D rules: a 1-D operand is **temporarily promoted** to 2-D, the matmul happens, then the
added axis is **removed again**. That is why `(4,3) @ (3,)` gives `(4,)` and not `(4,1)`.

The last line is the one worth studying: `(2,1,4,3) @ (5,3,6) -> (2,5,4,6)`. The *last two*
axes do the matmul (`3` vanishes); everything in front **broadcasts by 01.3's rules** —
`(2,1)` against `(5,)` padded to `(1,5)` gives `(2,5)`.

`torch.bmm` is the strict version: exactly 3-D, no broadcasting, no promotion. Use it when you
want the shape contract enforced.

---
# 01.7 — einsum, where the shape contract is written down

Rule: **a letter appearing in the inputs but not in the output is summed over.** Every letter
that survives into the output is an axis of the result.

> **Predict first:** in `"bhtd,bhsd->bhts"`, which letter is summed over, and how can you tell
> from the notation alone?
>
> _(write your answer here)_

In [8]:
a = torch.randn(3)
M, N = torch.randn(4, 3), torch.randn(3, 5)
Q = torch.randn(2, 2, 4, 8)      # (B, H, T, d_head)
K = torch.randn(2, 2, 4, 8)
V = torch.randn(2, 2, 4, 8)

checks = [
    ('dot product    ', torch.einsum('i,i->', a, a),                torch.dot(a, a)),
    ('matmul         ', torch.einsum('ij,jk->ik', M, N),            M @ N),
    ('attn scores    ', torch.einsum('bhtd,bhsd->bhts', Q, K),      Q @ K.transpose(-2, -1)),
    ('weighted values', torch.einsum('bhts,bhsd->bhtd', Q @ K.transpose(-2, -1), V),
                        (Q @ K.transpose(-2, -1)) @ V),
]
for name, e, direct in checks:
    shape = tuple(e.shape) if e.ndim else 'scalar'
    print(f'{name} {str(shape):16s} matches @ version: {torch.allclose(e, direct, atol=1e-5)}')

dot product     scalar           matches @ version: True
matmul          (4, 5)           matches @ version: True
attn scores     (2, 2, 4, 4)     matches @ version: True
weighted values (2, 2, 4, 8)     matches @ version: True


### Reading the output
Every one matches. Now compare the two ways of writing attention scores:

```python
Q @ K.transpose(-2, -1)              # which axes? you have to work it out
einsum('bhtd,bhsd->bhts', Q, K)      # batch, head, query-time, key-time, contract over d
```

The second **writes the contract down**. `d` appears in both inputs and not in the output, so
`d` is summed. `t` and `s` are both time axes, deliberately given different letters so you can
see that one indexes queries and the other keys.

`"bhtd,bhsd->bhtsd"` would keep `d` instead of summing it — a `(2,2,4,4,8)` tensor, eight times
larger, and not attention at all.

In [9]:
# break one index letter on purpose and read the error
try:
    torch.einsum('bhtd,bhsd->bhtz', Q, K)      # z never appeared in the inputs
except RuntimeError as e:
    print('unknown output letter ->', str(e).split('\n')[0][:100])

try:
    torch.einsum('bhtd,bhsd->bhts', Q, torch.randn(2, 2, 4, 9))   # d is 8 vs 9
except RuntimeError as e:
    print('\nmismatched contracted dim ->', str(e).split('\n')[0][:110])

unknown output letter -> einsum(): output subscript z does not appear in the equation for any input operand

mismatched contracted dim -> einsum(): subscript d has size 9 for operand 1 which does not broadcast with previously seen size 8


### Reading the output
Both are **loud errors**, and both name the offending letter.

That is einsum's real value here — not speed. A typo'd index becomes an exception instead of a
silent broadcast producing a plausible tensor, which is the failure mode of the whole 01.4 block.

---
# 01.8 — Reductions and the meaning of `dim=`

The question people get backwards: when you pass `dim=1`, are you reducing *along* that axis,
or *keeping* it?

> **Predict first:** `x` is `(2, 3, 4)`. What shape is `x.sum(dim=1)`?
>
> _(write your answer here)_

In [10]:
x = torch.arange(24).float().reshape(2, 3, 4)

for d in (0, 1, 2, -1):
    print(f'x.sum(dim={d:2d})                -> {tuple(x.sum(dim=d).shape)}')
print()
for d in (0, 1, 2):
    print(f'x.sum(dim={d}, keepdim=True)   -> {tuple(x.sum(dim=d, keepdim=True).shape)}')
print()
print('x.sum()                      ->', tuple(x.sum().shape), '(scalar — every axis gone)')
print('x.sum(dim=(0, 2))            ->', tuple(x.sum(dim=(0, 2)).shape))

x.sum(dim= 0)                -> (3, 4)
x.sum(dim= 1)                -> (2, 4)
x.sum(dim= 2)                -> (2, 3)
x.sum(dim=-1)                -> (2, 3)

x.sum(dim=0, keepdim=True)   -> (1, 3, 4)
x.sum(dim=1, keepdim=True)   -> (2, 1, 4)
x.sum(dim=2, keepdim=True)   -> (2, 3, 1)

x.sum()                      -> () (scalar — every axis gone)
x.sum(dim=(0, 2))            -> (3,)


### Reading the output
**The axis you name is the axis that disappears.** `dim=1` on `(2,3,4)` gives `(2,4)` — the `3`
is gone, because you summed *across* it.

`keepdim=True` leaves it behind as a `1`, giving `(2,1,4)`. That `1` is there for exactly one
reason: so the result still **broadcasts back** against the original. This is the fix for the
bug in the previous notebook.

`dim=-1` means the last axis, and it is the one you will write most — softmax, row maxima,
logits over vocabulary all reduce over the last axis.

In [11]:
# the pattern you will write in every attention implementation
scores = torch.randn(2, 2, 4, 4)                      # (B, H, T, T)

mx = scores.max(dim=-1, keepdim=True).values          # (B, H, T, 1)
assert_shape(mx, (2, 2, 4, 1), 'row max')

stable = scores - mx                                  # broadcasts back to (B,H,T,T)
probs  = stable.exp() / stable.exp().sum(dim=-1, keepdim=True)

print('scores     ', tuple(scores.shape))
print('row max    ', tuple(mx.shape), ' <- keepdim kept the 1')
print('probs      ', tuple(probs.shape))
print('rows sum to 1?', torch.allclose(probs.sum(dim=-1), torch.ones(2, 2, 4)))
print('matches torch.softmax?', torch.allclose(probs, torch.softmax(scores, dim=-1)))

scores      (2, 2, 4, 4)
row max     (2, 2, 4, 1)  <- keepdim kept the 1
probs       (2, 2, 4, 4)
rows sum to 1? True
matches torch.softmax? True


### Reading the output
You have just written a numerically-stable softmax over a `(B, H, T, T)` tensor, and it matches
`torch.softmax` exactly. Every `keepdim=True` in there is load-bearing — drop either one and the
shapes still broadcast, silently, into nonsense.

03.3 is the lesson that explains *why* subtracting the max is free. You have now built the shape
machinery it needs.

---
## Challenge — predict first, then run

| # | Question | Your answer |
| --- | --- | --- |
| 1 | `(2,3,4)` tensor, `.transpose(0,2)` — shape and stride? | _(write)_ |
| 2 | Does `.view(4,6)` work on that transposed tensor? | _(write)_ |
| 3 | `(3,)` @ `(3,5)` — output shape? | _(write)_ |
| 4 | `(8,1,4,5)` @ `(3,5,2)` — output shape? | _(write)_ |
| 5 | `einsum('ij,jk->ki', A, B)` with `A (2,3)`, `B (3,4)` — what shape, and how does it differ from `A @ B`? | _(write)_ |
| 6 | `x (2,3,4)`, `x.mean(dim=(1,2), keepdim=True)` — shape? | _(write)_ |

And in words: you are splitting `(B=4, T=128, d_model=512)` into 8 heads. Write the two
operations in order, with the shape after each.

In [ ]:
# run AFTER writing your predictions above
z = torch.randn(2, 3, 4).transpose(0, 2)
print('1.', tuple(z.shape), z.stride(), ' contiguous', z.is_contiguous())
try:
    z.view(4, 6); print('2. view worked')
except RuntimeError:
    print('2. view FAILED')
print('3.', tuple(torch.matmul(torch.randn(3), torch.randn(3, 5)).shape))
print('4.', tuple(torch.matmul(torch.randn(8, 1, 4, 5), torch.randn(3, 5, 2)).shape))
A2, B2 = torch.randn(2, 3), torch.randn(3, 4)
print('5.', tuple(torch.einsum('ij,jk->ki', A2, B2).shape), 'vs A @ B', tuple((A2 @ B2).shape))
print('6.', tuple(torch.randn(2, 3, 4).mean(dim=(1, 2), keepdim=True).shape))